<div style="background:linear-gradient(120deg,#071225,#1e285a,#48276c);padding:36px;border-radius:18px;color:white">
<p style="color:#67e8f9;letter-spacing:3px">MARKET DATA LAB • 2026</p>
<h1 style="font-size:38px;color:white">SpaceX / SPCX</h1>
<h2 style="color:#c4b5fd">From IPO history to honest forecasts</h2>
<p>Exploratory analysis · risk · chronological model evaluation · year-end scenarios</p>
<p><b>Muhammad Ibrahim Shahrukh</b><br>Snapshot: June 12–October 19, 2026 · Daily OHLCV · USD</p>
</div>

**Question:** What can 89 observations tell us about SPCX, and how much confidence should we place in a forecast to December 31, 2026?

This notebook answers that question in stages: inspect → explore → evaluate → forecast → challenge the result.
There is no universal best stock model. We select the best of a small, declared candidate set on earlier validation data and assess it once on a later holdout.

> **Scope:** An educational statistical forecast from the supplied snapshot, not a current quote or investment recommendation. Adjustment status is unknown; returns are changes in the supplied close series, not verified total returns. The end-of-year horizon is longer than the tested horizon.

**Contents:** 01 Setup • 02 Quality • 03 Price & volume • 04 Returns & risk • 05 Patterns • 06 Forecast design • 07 Validation • 08 Holdout • 09 Year-end scenarios • 10 Exports & sources

<div style="background:#111d35;border-left:6px solid #22d3ee;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#22d3ee;font-weight:bold">01</span><h2 style="margin:6px 0;color:#fff">Make the notebook portable</h2><p style="margin:0">Run all cells in order. Attach spcx_daily_clean.csv in Kaggle; no live download or internet access is needed.</p></div>

In [1]:
from pathlib import Path
import json, sys, platform, importlib.metadata, warnings
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, HTML, Markdown
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
DATA_PATH = None  # Optional: Path('/kaggle/input/your-dataset/spcx_daily_clean.csv')
OUTPUT_DIR = Path('/kaggle/working/spcx_outputs') if Path('/kaggle/working').exists() else Path('reports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
END_DATE = pd.Timestamp('2026-12-31')
HOLIDAYS = pd.to_datetime(['2026-01-01','2026-01-19','2026-02-16','2026-04-03',
    '2026-05-25','2026-06-19','2026-07-03','2026-09-07','2026-11-26','2026-12-25'])
def sessions(start, end):
    dates = pd.bdate_range(start, end)
    return dates[~dates.isin(HOLIDAYS)]

COLORS = ['#22d3ee','#a78bfa','#34d399','#fb7185','#fbbf24','#60a5fa','#f472b6','#94a3b8']
pio.templates.default = 'plotly_dark'
pd.options.display.float_format = '{:,.4f}'.format
FIGURES = []
def finish(fig, title, height=470):
    fig.update_layout(title=dict(text=title, x=0.03), height=height,
        paper_bgcolor='#0b1327', plot_bgcolor='#111d35', colorway=COLORS,
        font=dict(family='Arial',color='#e2e8f0',size=12),
        margin=dict(l=60,r=35,t=85,b=60), legend=dict(orientation='h',y=-0.2))
    FIGURES.append(fig)
    return fig

print('Python:', platform.python_version())
print({p:importlib.metadata.version(p) for p in ['numpy','pandas','plotly','scikit-learn','scipy']})

Python: 3.12.14
{'numpy': '2.3.5', 'pandas': '2.2.3', 'plotly': '7.1.0', 'scikit-learn': '1.8.0', 'scipy': '1.17.0'}


In [2]:
if DATA_PATH is None:
    candidates = [p for p in [Path('data/spcx_daily_clean.csv'), Path('spcx_daily_clean.csv')]
                  if p.is_file()]
    if not candidates and Path('/kaggle/input').exists():
        candidates = sorted(Path('/kaggle/input').rglob('spcx_daily_clean.csv'))
    candidates = list({p.resolve(): p for p in candidates}.values())
    if len(candidates) != 1:
        raise ValueError(f'Expected one clean CSV; found {len(candidates)}. Set DATA_PATH explicitly.')
    DATA_PATH = candidates[0]
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
assert df.columns.tolist() == ['date','ticker','open','high','low','close','volume']
assert len(df) >= 80, 'This evaluation design requires at least 80 observations.'
assert df.date.dt.year.eq(2026).all(), 'The embedded session calendar is specific to 2026.'
assert df.date.is_unique and df.date.is_monotonic_increasing
assert df.ticker.eq('SPCX').all() and not df.isna().any().any()
assert np.isfinite(df.select_dtypes('number')).all().all()
assert (df[['open','high','low','close']] > 0).all().all()
assert (df.volume >= 0).all() and (df.volume % 1 == 0).all()
assert (df.high >= df[['open','low','close']].max(axis=1)).all()
assert (df.low <= df[['open','high','close']].min(axis=1)).all()
missing_sessions = sessions(df.date.min(), df.date.max()).difference(df.date)
unexpected_sessions = pd.DatetimeIndex(df.date).difference(sessions(df.date.min(), df.date.max()))
assert len(missing_sessions) == 0 and len(unexpected_sessions) == 0, 'Review calendar discrepancies.'
display(df.head())
print(f'Loaded {len(df)} rows: {df.date.min().date()} to {df.date.max().date()}')

Loaded 89 rows: 2026-06-12 to 2026-10-19


,date,ticker,open,high,low,close,volume
0,2026-06-12,SPCX,150.0000,176.5200,149.3400,160.9500,519234800
1,2026-06-15,SPCX,171.7400,193.0000,168.3500,192.5000,256226600
2,2026-06-16,SPCX,200.5100,225.6400,199.9800,211.3900,195401600
3,2026-06-17,SPCX,209.8400,213.8000,187.0100,191.8200,201719500
4,2026-06-18,SPCX,188.3900,190.0000,172.1100,185.0000,272126800


<div style="background:#111d35;border-left:6px solid #34d399;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#34d399;font-weight:bold">02</span><h2 style="margin:6px 0;color:#fff">Data quality & provenance</h2><p style="margin:0">Each row is one trading session. Weekends and scheduled market holidays are absent by design.</p></div>

The uploader reports collecting this file from **Yahoo Finance using yfinance**. Its two column-header levels and date-label row match a yfinance MultiIndex CSV export. The original command, download timestamp, package version, timezone settings and `auto_adjust` setting were not supplied. We do not infer them from the file.

**Refinement:** remove metadata header rows; rename the date column; flatten column names; add `ticker`; parse dates and numeric values; sort chronologically; check duplicates, missing data and OHLC bounds; store volume as an integer; round prices to two decimals. No interpolation, synthetic history, outlier deletion, corporate-action adjustment or currency conversion was performed. Exact duplicates would be removed; conflicting duplicate dates stop the cleaner.

**Ticker check:** [Nasdaq's June 12, 2026 listing report](https://www.nasdaq.com/newsroom/spacex-ipo-rocket-company-launches-historic-ipo) identifies SpaceX as SPCX. The uploaded history starts on that date. Do not join old data for a different security just because it used the same ticker. Individual historical bars were not independently reconciled to an exchange feed.

| Column | Short description | Type / unit |
|---|---|---|
| date | Trading session date | YYYY-MM-DD |
| ticker | Stock ticker symbol | SPCX |
| open | Opening share price | USD/share |
| high | Highest session price | USD/share |
| low | Lowest session price | USD/share |
| close | Closing share price | USD/share |
| volume | Shares traded | Integer shares |

In [3]:
quality = pd.DataFrame({
    'check':['Rows','Missing cells','Duplicate dates','Missing scheduled sessions','Unexpected sessions','Zero-volume rows'],
    'value':[len(df),int(df.isna().sum().sum()),int(df.date.duplicated().sum()),len(missing_sessions),
             len(unexpected_sessions),int(df.volume.eq(0).sum())]})
display(quality)
quality.to_csv(OUTPUT_DIR/'quality_checks.csv',index=False)
display(df.drop(columns=['ticker']).describe().T)

eda = df.copy()
eda['return_pct'] = eda.close.pct_change(fill_method=None)*100
eda['log_return'] = np.log(eda.close).diff()
eda['drawdown_pct'] = (eda.close/eda.close.cummax()-1)*100
eda['range_pct'] = (eda.high-eda.low)/eda.open*100
eda['gap_pct'] = (eda.open/eda.close.shift(1)-1)*100
eda['intraday_pct'] = (eda.close/eda.open-1)*100
eda['vol_20_pct'] = eda.log_return.rolling(20).std()*np.sqrt(252)*100
eda['sma_10'] = eda.close.rolling(10).mean()
eda['sma_20'] = eda.close.rolling(20).mean()
eda['weekday'] = eda.date.dt.day_name()
eda['volume_m'] = eda.volume/1e6
eda['abs_return_pct'] = eda.return_pct.abs()
eda['volume_change_pct'] = eda.volume.pct_change(fill_method=None)*100
total_return = (df.close.iloc[-1]/df.close.iloc[0]-1)*100
metrics = [('OBSERVATIONS',str(len(df))),('LAST CLOSE',f'${df.close.iloc[-1]:,.2f}'),
           ('CLOSE CHANGE',f'{total_return:+.2f}%'),('MAX DRAWDOWN',f'{eda.drawdown_pct.min():.2f}%')]
cards=''.join(f'<div style="display:inline-block;min-width:175px;margin:5px;padding:18px;background:#17243f;border-top:4px solid {COLORS[i]};border-radius:10px"><div style="color:#94a3b8;font-size:11px">{k}</div><div style="color:{COLORS[i]};font-size:27px;font-weight:bold">{v}</div></div>' for i,(k,v) in enumerate(metrics))
display(HTML('<div>'+cards+'</div>'))

,check,value
0,Rows,89
1,Missing cells,0
2,Duplicate dates,0
3,Missing scheduled sessions,0
4,Unexpected sessions,0
5,Zero-volume rows,0


,count,mean,min,25%,50%,75%,max,std
date,89,2026-08-16 09:10:06.741573120,2026-06-12 00:00:00,2026-07-16 00:00:00,2026-08-17 00:00:00,2026-09-17 00:00:00,2026-10-19 00:00:00,NaN
open,89.0000,145.4883,106.4000,137.0000,147.8000,154.2000,209.8400,19.1451
high,89.0000,150.0601,113.6400,139.7500,150.5700,159.3000,225.6400,20.1039
low,89.0000,141.1601,104.8300,133.1000,145.0700,150.0000,199.9800,17.8913
close,89.0000,145.6429,108.2700,136.9700,147.9600,154.5400,211.3900,18.7130
volume,89.0000,"104,071,847.1910","46,468,600.0000","61,380,500.0000","79,718,100.0000","118,598,900.0000","519,234,800.0000","72,208,051.4183"


<div style="background:#111d35;border-left:6px solid #22d3ee;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#22d3ee;font-weight:bold">03</span><h2 style="margin:6px 0;color:#fff">The price journey</h2><p style="margin:0">Candles preserve the open, high, low and close. Volume adds context but does not explain causality.</p></div>

In [4]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.72,0.28])
fig.add_trace(go.Candlestick(x=eda.date,open=eda.open,high=eda.high,low=eda.low,close=eda.close,
    name='OHLC',increasing_line_color='#34d399',decreasing_line_color='#fb7185'),row=1,col=1)
for col,name,color in [('sma_10','10-session mean','#22d3ee'),('sma_20','20-session mean','#fbbf24')]:
    fig.add_trace(go.Scatter(x=eda.date,y=eda[col],name=name,line=dict(color=color,width=2)),row=1,col=1)
fig.add_trace(go.Bar(x=eda.date,y=eda.volume_m,name='Volume (million)',marker_color='#a78bfa'),row=2,col=1)
fig.update_layout(xaxis_rangeslider_visible=False)
fig.update_yaxes(title_text='USD/share',row=1,col=1)
fig.update_yaxes(title_text='Million shares',row=2,col=1)
finish(fig,'01 • Price structure and trading activity',650)
fig.show(renderer="iframe")

**How to read this:** moving averages smooth past prices; they are not forecasts. Early activity may differ from later activity after an IPO. Keep the large moves in the data unless they can be verified as errors. Initial missing rolling values are expected warm-up periods, not defects in the clean CSV.

<div style="background:#111d35;border-left:6px solid #fb7185;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#fb7185;font-weight:bold">04</span><h2 style="margin:6px 0;color:#fff">Returns, drawdowns & volatility</h2><p style="margin:0">Price changes and losses from a previous closing peak reveal different aspects of risk.</p></div>

In [5]:
fig=make_subplots(rows=3,cols=1,shared_xaxes=True,vertical_spacing=0.09,
    subplot_titles=['Daily close-to-close return','Drawdown from running closing peak','20-session annualized log-return volatility'])
fig.add_trace(go.Bar(x=eda.date,y=eda.return_pct,name='Daily return',marker_color=np.where(eda.return_pct>=0,'#34d399','#fb7185')),row=1,col=1)
fig.add_trace(go.Scatter(x=eda.date,y=eda.drawdown_pct,fill='tozeroy',name='Drawdown',line_color='#fb7185'),row=2,col=1)
fig.add_trace(go.Scatter(x=eda.date,y=eda.vol_20_pct,name='Rolling volatility',line_color='#fbbf24'),row=3,col=1)
fig.update_yaxes(ticksuffix='%')
finish(fig,'02 • The risk dashboard',760)
fig.show(renderer="iframe")

rets=eda.return_pct.dropna()
risk=pd.DataFrame({'measure':['Close-to-close change (%)','Max closing drawdown (%)','Daily return std (%)',
    'Annualized log volatility (%)','Best daily return (%)','Worst daily return (%)','Positive sessions (%)'],
    'value':[total_return,eda.drawdown_pct.min(),rets.std(),eda.log_return.std()*np.sqrt(252)*100,
             rets.max(),rets.min(),rets.gt(0).mean()*100]})
display(risk)
risk.to_csv(OUTPUT_DIR/'risk_summary.csv',index=False)

,measure,value
0,Close-to-close change (%),-8.0708
1,Max closing drawdown (%),-48.7819
2,Daily return std (%),5.2201
3,Annualized log volatility (%),82.2675
4,Best daily return (%),19.6024
5,Worst daily return (%),-16.4324
6,Positive sessions (%),45.4545


Volatility annualization uses 252 sessions as a convention; it is not a forecast of next year's risk. Drawdown uses closing prices and starts at the first observed close, so it excludes intraday and pre-listing losses. No CAGR or long-term seasonal claim is warranted from this short sample.

In [6]:
theory, ordered = stats.probplot(rets, dist='norm', fit=False)
fig=make_subplots(rows=1,cols=2,subplot_titles=['Return distribution','Normal Q–Q diagnostic'])
fig.add_trace(go.Histogram(x=rets,nbinsx=18,name='Daily returns',marker_color='#22d3ee'),row=1,col=1)
fig.add_trace(go.Scatter(x=theory,y=ordered,mode='markers',name='Observed quantiles',marker_color='#a78bfa'),row=1,col=2)
line=np.asarray(theory)*rets.std()+rets.mean()
fig.add_trace(go.Scatter(x=theory,y=line,name='Normal reference',line=dict(color='#fbbf24',dash='dash')),row=1,col=2)
fig.update_xaxes(title_text='Daily return (%)',row=1,col=1)
fig.update_xaxes(title_text='Theoretical normal quantile',row=1,col=2)
fig.update_yaxes(title_text='Count',row=1,col=1)
fig.update_yaxes(title_text='Daily return (%)',row=1,col=2)
finish(fig,'03 • Tail behavior: check the normal assumption')
fig.show(renderer="iframe")

<div style="background:#111d35;border-left:6px solid #a78bfa;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#a78bfa;font-weight:bold">05</span><h2 style="margin:6px 0;color:#fff">Explore patterns without overclaiming</h2><p style="margin:0">Short samples can produce persuasive patterns by chance. Treat these panels as descriptive evidence.</p></div>

In [7]:
month=eda.assign(month=eda.date.dt.to_period('M').astype(str)).groupby('month').agg(
    return_pct=('return_pct',lambda x:((1+x.dropna()/100).prod()-1)*100),sessions=('date','size'))
labels=[m+(' (partial)' if m in [eda.date.min().strftime('%Y-%m'),eda.date.max().strftime('%Y-%m')] else '') for m in month.index]
fig=go.Figure(go.Heatmap(z=[month.return_pct.tolist()],x=labels,y=['Observed return'],
    text=[[f'{v:+.2f}%<br>n={n}' for v,n in zip(month.return_pct,month.sessions)]],texttemplate='%{text}',
    colorscale='RdBu',zmid=0,colorbar=dict(title='%')))
finish(fig,'04 • Compounded returns by observed month',330)
fig.show(renderer="iframe")

In [8]:
display(month)

weekdays=['Monday','Tuesday','Wednesday','Thursday','Friday']
fig=px.box(eda.dropna(subset=['return_pct']),x='weekday',y='return_pct',color='weekday',points='all',
    category_orders={'weekday':weekdays},labels={'return_pct':'Daily return (%)','weekday':'Trading weekday'})
finish(fig,'05 • Weekday distributions — exploratory only')
fig.show(renderer="iframe")

,return_pct,sessions
month,,
2026-06,6.1572,12
2026-07,-36.5738,22
2026-08,32.5920,21
2026-09,4.9899,21
2026-10,-1.9223,13


In [9]:
display(eda.groupby('weekday').return_pct.agg(['count','mean','median','std']).reindex(weekdays))

,count,mean,median,std
weekday,,,,
Monday,18,1.0893,-0.0202,7.1117
Tuesday,18,0.9726,1.4334,4.4081
Wednesday,18,-2.3827,-2.5435,5.2212
Thursday,18,0.0331,-0.2667,3.3007
Friday,16,0.5303,-0.3813,5.0793


June begins at the first observed close, and October stops on October 19. These are partial periods. Full months compound daily returns from the preceding observed month-end close. Weekday averages are based on very few observations, with no multiple-testing correction or causal interpretation. They are not trading rules.

In [10]:
fig=px.scatter(eda.dropna(subset=['return_pct']),x='volume_m',y='abs_return_pct',color='range_pct',
    hover_data=['date','close'],color_continuous_scale='Plasma',
    labels={'volume_m':'Volume (million shares)','abs_return_pct':'Absolute daily return (%)','range_pct':'Intraday range (%)'})
finish(fig,'06 • Does trading activity accompany larger moves?')
fig.show(renderer="iframe")

In [11]:
corrcols=['return_pct','gap_pct','intraday_pct','range_pct','volume_change_pct']
corr=eda[corrcols].corr(method='spearman')
fig=px.imshow(corr,text_auto='.2f',zmin=-1,zmax=1,color_continuous_scale='RdBu',aspect='auto')
finish(fig,'07 • Spearman associations among changes',530)
fig.show(renderer="iframe")

In [12]:
lags=np.arange(1,11)
acf=[eda.log_return.dropna().autocorr(int(l)) for l in lags]
fig=go.Figure(go.Bar(x=lags,y=acf,marker_color='#22d3ee',name='Sample autocorrelation'))
bound=1.96/np.sqrt(eda.log_return.notna().sum())
fig.add_hline(y=bound,line_dash='dash',line_color='#fbbf24')
fig.add_hline(y=-bound,line_dash='dash',line_color='#fbbf24')
fig.update_xaxes(title='Lag in trading sessions',dtick=1)
fig.update_yaxes(title='Log-return autocorrelation')
finish(fig,'08 • Return memory — approximate white-noise reference')
fig.show(renderer="iframe")

The autocorrelation reference is approximate, assumes white noise, and is not adjusted for testing many lags. Correlations among returns avoid the most obvious shared-price-level trend problem but still do not establish predictive power. EDA uses the full snapshot for description; the candidate definitions and model selection rules below are fixed and do not use the holdout scores.

<div style="background:#111d35;border-left:6px solid #34d399;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#34d399;font-weight:bold">06</span><h2 style="margin:6px 0;color:#fff">A fair forecasting experiment</h2><p style="margin:0">The best available model here means the best validated candidate—not a promise of future accuracy.</p></div>

**Target:** next-session log return, recursively compounded into future closing prices. Predictors are five recent log returns, five- and ten-session mean returns, and five- and ten-session return standard deviations. All are available at the forecast origin. Future volume, open, high and low are unknown, so they are not predictors.

**Candidates:** last-close (random walk), mean log-return drift, damped drift, Ridge with three fixed penalties, shallow Random Forest and Extra Trees. Neural networks and large searches would be difficult to justify with 89 observations.

**Protocol:**
1. Reserve the last 20 sessions as one untouched, fixed-origin holdout.
2. In the earlier history, fit at three expanding origins and forecast each next 10-session block without seeing that block's realized prices.
3. Select the lowest mean validation price MAE; break exact ties by RMSE. All preprocessing is fitted inside each training window.
4. Fit the selected configuration to all pre-holdout data and evaluate once over 20 future sessions. Show all candidates for transparency, but do not reselect on this table.
5. Refit the already-selected configuration to the complete supplied history for the year-end forecast.

**Metrics:** MAE and RMSE in USD (lower is better); MAPE in percent; MASE relative to training one-step absolute price changes; R², which can be negative; and horizon-direction agreement relative to the last known close. MAE skill compares each path to the last-close baseline at the same origin. MASE's scaling is one-step, so it is not a same-horizon benchmark. Holdout results describe one path, not repeated independent trials.

In [13]:
FEATURES=['return_lag_1','return_lag_2','return_lag_3','return_lag_4','return_lag_5',
          'mean_5','mean_10','std_5','std_10']
def feature_row(history):
    a=np.asarray(history,dtype=float)
    assert len(a)>=10
    return np.r_[a[-1:-6:-1],a[-5:].mean(),a[-10:].mean(),a[-5:].std(ddof=1),a[-10:].std(ddof=1)]
def training_matrix(prices):
    r=np.diff(np.log(np.asarray(prices,dtype=float)))
    X=np.array([feature_row(r[:t]) for t in range(10,len(r))])
    return X,r[10:]
def estimator(name):
    if name.startswith('Ridge'):
        return make_pipeline(StandardScaler(),Ridge(alpha=float(name.split('_')[1])))
    if name=='RandomForest':
        return RandomForestRegressor(n_estimators=160,max_depth=3,min_samples_leaf=5,random_state=SEED,n_jobs=1)
    if name=='ExtraTrees':
        return ExtraTreesRegressor(n_estimators=160,max_depth=3,min_samples_leaf=5,random_state=SEED,n_jobs=1)
    raise ValueError(name)

MODELS=['LastClose','LogDrift','DampedDrift','Ridge_1','Ridge_10','Ridge_100','RandomForest','ExtraTrees']
def predict_path(name,train,horizon):
    train=np.asarray(train,dtype=float)
    r=list(np.diff(np.log(train)))
    fitted=None
    if name=='LastClose':
        future=np.zeros(horizon)
    elif name=='LogDrift':
        future=np.repeat(np.mean(r),horizon)
    elif name=='DampedDrift':
        future=np.mean(r)*0.9**np.arange(1,horizon+1)
    else:
        X,y=training_matrix(train)
        fitted=estimator(name).fit(X,y)
        future=[]
        for _ in range(horizon):
            pred=float(fitted.predict(feature_row(r).reshape(1,-1))[0])
            future.append(pred)
            r.append(pred)
    prices=train[-1]*np.exp(np.cumsum(future))
    assert np.isfinite(prices).all() and (prices>0).all()
    return prices,fitted

def scores(actual,pred,train):
    actual,pred,train=map(np.asarray,[actual,pred,train])
    scale=np.abs(np.diff(train)).mean()
    base=np.repeat(train[-1],len(actual))
    baseline_mae=mean_absolute_error(actual,base)
    mae=mean_absolute_error(actual,pred)
    return {'MAE_USD':mae,'RMSE_USD':np.sqrt(mean_squared_error(actual,pred)),
            'MAPE_pct':np.mean(np.abs((actual-pred)/actual))*100,'MASE':mae/scale,
            'R2':r2_score(actual,pred),
            'Horizon_direction_pct':np.mean(np.sign(actual-train[-1])==np.sign(pred-train[-1]))*100,
            'MAE_skill_pct':100*(1-mae/baseline_mae) if baseline_mae>0 else np.nan}

values=df.close.to_numpy()
HOLDOUT=20
dev=values[:-HOLDOUT]
origins=[len(dev)-30,len(dev)-20,len(dev)-10]
assert min(origins)>=35
fold_rows=[]
for fold,origin in enumerate(origins,1):
    fold_rows.append({'fold':fold,'train_rows':origin,'train_end':df.date.iloc[origin-1].date(),
        'validation_start':df.date.iloc[origin].date(),'validation_end':df.date.iloc[origin+9].date(),
        'supervised_train_rows':len(training_matrix(dev[:origin])[1])})
display(pd.DataFrame(fold_rows))
print('Holdout:',df.date.iloc[-HOLDOUT].date(),'through',df.date.iloc[-1].date())

Holdout: 2026-09-22 through 2026-10-19


,fold,train_rows,train_end,validation_start,validation_end,supervised_train_rows
0,1,39,2026-08-07,2026-08-10,2026-08-21,28
1,2,49,2026-08-21,2026-08-24,2026-09-04,38
2,3,59,2026-09-04,2026-09-08,2026-09-21,48


<div style="background:#111d35;border-left:6px solid #22d3ee;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#22d3ee;font-weight:bold">07</span><h2 style="margin:6px 0;color:#fff">Expanding-window validation</h2><p style="margin:0">Each validation path is generated without access to the next ten observed closes.</p></div>

In [14]:
cv_rows=[]
cv_paths=[]
for fold,origin in enumerate(origins,1):
    train=dev[:origin]
    actual=dev[origin:origin+10]
    for name in MODELS:
        pred,_=predict_path(name,train,10)
        cv_rows.append({'model':name,'fold':fold,**scores(actual,pred,train)})
        for h,(a,p) in enumerate(zip(actual,pred),1):
            cv_paths.append({'model':name,'fold':fold,'horizon':h,'actual':a,'prediction':p})
cv=pd.DataFrame(cv_rows)
ranking=cv.groupby('model').agg(MAE_USD=('MAE_USD','mean'),RMSE_USD=('RMSE_USD','mean'),
    fold_MAE_std=('MAE_USD','std'),MAPE_pct=('MAPE_pct','mean'),MASE=('MASE','mean')).sort_values(['MAE_USD','RMSE_USD'])
SELECTED=ranking.index[0]  # Frozen before examining the holdout.
display(ranking)
print('Selected on validation:',SELECTED)
fig=px.bar(ranking.reset_index(),x='model',y='MAE_USD',error_y='fold_MAE_std',color='model',
    labels={'MAE_USD':'Mean validation MAE (USD)','model':'Candidate'})
finish(fig,'09 • Validation ranking — bars show fold-to-fold variation',540)
fig.show(renderer="iframe")
cv.to_csv(OUTPUT_DIR/'validation_fold_metrics.csv',index=False)
pd.DataFrame(cv_paths).to_csv(OUTPUT_DIR/'validation_predictions.csv',index=False)
ranking.to_csv(OUTPUT_DIR/'validation_ranking.csv')

Selected on validation: LastClose


,MAE_USD,RMSE_USD,fold_MAE_std,MAPE_pct,MASE
model,,,,,
LastClose,5.1530,6.1570,1.8101,3.5536,0.7290
DampedDrift,6.6908,7.6594,2.7662,4.6316,0.9428
LogDrift,7.4348,8.4614,3.1930,5.1526,1.0465
ExtraTrees,7.8063,8.9978,4.8267,5.4250,1.0942
RandomForest,8.7288,9.9704,5.3988,6.0569,1.2491
Ridge_100,9.7147,11.2491,4.1124,6.7023,1.3879
Ridge_10,10.7425,12.7541,9.0655,7.4223,1.5387
Ridge_1,11.9418,13.8346,9.7541,8.3285,1.6900


The error bars show the standard deviation across only three folds, not a confidence interval. Selecting among multiple models on a small validation set can still overfit the model-selection decision. A baseline winning is a useful result; an ML label does not make a forecast better.

<div style="background:#111d35;border-left:6px solid #fbbf24;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#fbbf24;font-weight:bold">08</span><h2 style="margin:6px 0;color:#fff">The untouched holdout</h2><p style="margin:0">Twenty future sessions, one fixed origin, and no model reselection.</p></div>

In [15]:
test=values[-HOLDOUT:]
test_dates=df.date.iloc[-HOLDOUT:]
holdout_rows=[]
holdout_paths={}
for name in MODELS:
    pred,_=predict_path(name,dev,HOLDOUT)
    holdout_paths[name]=pred
    holdout_rows.append({'model':name,'selected_on_validation':name==SELECTED,**scores(test,pred,dev)})
holdout=pd.DataFrame(holdout_rows).set_index('model').loc[ranking.index]
display(holdout)
selected_test=holdout_paths[SELECTED]
fig=go.Figure()
fig.add_trace(go.Scatter(x=test_dates,y=test,name='Observed',line=dict(color='#fff',width=3)))
fig.add_trace(go.Scatter(x=test_dates,y=selected_test,name=f'Selected: {SELECTED}',line=dict(color='#22d3ee',width=3)))
if SELECTED!='LastClose':
    fig.add_trace(go.Scatter(x=test_dates,y=holdout_paths['LastClose'],name='Last-close baseline',line=dict(color='#fbbf24',dash='dash')))
fig.update_yaxes(title='Closing price (USD)')
finish(fig,'10 • Fixed-origin 20-session holdout')
fig.show(renderer="iframe")

,selected_on_validation,MAE_USD,RMSE_USD,MAPE_pct,MASE,R2,Horizon_direction_pct,MAE_skill_pct
model,,,,,,,,
LastClose,True,6.9115,8.9045,4.3073,1.1751,-0.1476,0.0000,0.0000
DampedDrift,False,6.9907,9.2557,4.3378,1.1886,-0.2399,50.0000,-1.1454
LogDrift,False,7.0796,9.6093,4.3771,1.2037,-0.3365,50.0000,-2.4328
ExtraTrees,False,7.3055,8.5178,4.6951,1.2421,-0.0501,50.0000,-5.7008
RandomForest,False,8.8649,11.4184,5.8184,1.5073,-0.8871,50.0000,-28.2625
Ridge_100,False,12.4985,15.6227,7.8143,2.1251,-2.5326,50.0000,-80.8357
Ridge_10,False,14.5355,17.8625,9.1324,2.4714,-3.6181,50.0000,-110.3094
Ridge_1,False,13.7481,16.8667,8.6351,2.3375,-3.1176,40.0000,-98.9163


In [16]:
holdout.to_csv(OUTPUT_DIR/'holdout_metrics.csv')
pd.DataFrame({'date':test_dates.to_numpy(),'actual':test,**holdout_paths}).to_csv(OUTPUT_DIR/'holdout_predictions.csv',index=False)

errors=test-selected_test
fig=make_subplots(rows=1,cols=2,subplot_titles=['Forecast error by horizon','Predicted vs. observed close'])
fig.add_trace(go.Bar(x=np.arange(1,HOLDOUT+1),y=errors,name='Actual minus prediction',marker_color='#fb7185'),row=1,col=1)
fig.add_trace(go.Scatter(x=selected_test,y=test,mode='markers',name='Holdout observations',marker=dict(color=np.arange(HOLDOUT),colorscale='Viridis',size=9)),row=1,col=2)
lo=min(test.min(),selected_test.min()); hi=max(test.max(),selected_test.max())
fig.add_trace(go.Scatter(x=[lo,hi],y=[lo,hi],name='Perfect prediction',line=dict(dash='dash',color='#fbbf24')),row=1,col=2)
fig.update_xaxes(title='Forecast horizon (sessions)',row=1,col=1)
fig.update_xaxes(title='Predicted USD',row=1,col=2)
fig.update_yaxes(title='Error USD',row=1,col=1)
fig.update_yaxes(title='Observed USD',row=1,col=2)
finish(fig,'11 • Diagnose where the selected forecast misses')
fig.show(renderer="iframe")

In [17]:
skill=float(holdout.loc[SELECTED,'MAE_skill_pct'])
display(Markdown(f'**Holdout interpretation:** `{SELECTED}` has MAE **${holdout.loc[SELECTED,"MAE_USD"]:.2f}**, RMSE **${holdout.loc[SELECTED,"RMSE_USD"]:.2f}**, and MAE skill **{skill:+.1f}%** versus last-close. Positive skill beats that baseline; negative skill loses to it. The selected model remains unchanged regardless of this result.'))

**Holdout interpretation:** `LastClose` has MAE **$6.91**, RMSE **$8.90**, and MAE skill **+0.0%** versus last-close. Positive skill beats that baseline; negative skill loses to it. The selected model remains unchanged regardless of this result.

Forecast errors at adjacent horizons are dependent. Negative R² means the forecast performs worse than the constant mean of the realized test values, a reference unavailable at forecast time. Horizon-direction agreement measures whether a future price is above or below the origin close; it is not daily trading accuracy. A last-close forecast predicts no change and receives no directional credit when the realized change is nonzero.

In [18]:
# Descriptive model inspection, fitted only on pre-holdout observations.
forest=estimator('RandomForest')
X_dev,y_dev=training_matrix(dev)
forest.fit(X_dev,y_dev)
importance=pd.DataFrame({'feature':FEATURES,'importance':forest.feature_importances_}).sort_values('importance')
fig=px.bar(importance,x='importance',y='feature',orientation='h',color='importance',color_continuous_scale='Teal')
finish(fig,'12 • Random Forest impurity importance — training diagnostic')
fig.show(renderer="iframe")

This Random Forest panel is an illustrative training diagnostic, even if another model wins. Impurity importance is biased and can split credit among correlated predictors. It is neither causal evidence nor a test of out-of-sample usefulness.

<div style="background:#111d35;border-left:6px solid #22d3ee;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#22d3ee;font-weight:bold">09</span><h2 style="margin:6px 0;color:#fff">Forecast the remaining 2026 sessions</h2><p style="margin:0">Refit on all supplied observations. Keep projections separate from historical records.</p></div>

Forecast dates begin after **October 19**, the final observed session. Dates after that snapshot remain forecasts even if a reader runs the notebook later. Weekends and Nasdaq's full-day holidays are excluded; November 27 and December 24 are included as early-close sessions. The calendar is explicitly scoped to 2026.

**Uncertainty method:** take the latest 60 observed log returns, center them at zero, resample circular blocks of five to retain some short-run dependence, and accumulate the shocks around the selected deterministic log-price path. The 10th–90th and 2.5th–97.5th percentiles are **80% and 95% bootstrap scenario bands**. They are not calibrated prediction intervals or guaranteed coverage; they omit uncertainty about model choice, drift, future volatility regimes and unobserved shocks. They are pointwise, not simultaneous path bands.

We also check this same procedure at the pre-holdout origin using only information then available. Twenty dependent observations are not enough to establish calibration. The year-end horizon is 51 sessions, compared with 10-session validation and a 20-session holdout; long-horizon accuracy has not been validated.

In [19]:
def scenario_bands(train,center,n_paths=4000,block=5,seed=SEED):
    r=np.diff(np.log(np.asarray(train,dtype=float)))[-60:]
    centered=r-r.mean()
    rng=np.random.default_rng(seed)
    horizon=len(center)
    starts=rng.integers(0,len(centered),size=(n_paths,int(np.ceil(horizon/block))))
    indices=(starts[:,:,None]+np.arange(block)[None,None,:])%len(centered)
    shocks=centered[indices].reshape(n_paths,-1)[:,:horizon]
    sims=np.asarray(center)[None,:]*np.exp(np.cumsum(shocks,axis=1))
    return np.quantile(sims,[0.025,0.10,0.90,0.975],axis=0)

test_bands=scenario_bands(dev,selected_test)
coverage80=np.mean((test>=test_bands[1])&(test<=test_bands[2]))*100
coverage95=np.mean((test>=test_bands[0])&(test<=test_bands[3]))*100
print(f'One-origin holdout scenario coverage: 80% band {coverage80:.0f}%; 95% band {coverage95:.0f}%. Not calibration evidence.')

future_dates=sessions(df.date.max()+pd.Timedelta(days=1),END_DATE)
assert len(future_dates)>0, 'No remaining 2026 sessions after the snapshot.'
center,final_model=predict_path(SELECTED,values,len(future_dates))
bands=scenario_bands(values,center)
forecast=pd.DataFrame({'date':future_dates,'ticker':'SPCX','model':SELECTED,
    'forecast_close':center,'scenario_low_80':bands[1],'scenario_high_80':bands[2],
    'scenario_low_95':bands[0],'scenario_high_95':bands[3],
    'forecast_origin':df.date.max().strftime('%Y-%m-%d'),'horizon_sessions':np.arange(1,len(future_dates)+1)})
assert (forecast.scenario_low_95<=forecast.scenario_low_80).all()
assert (forecast.scenario_low_80<=forecast.scenario_high_80).all()
assert (forecast.scenario_high_80<=forecast.scenario_high_95).all()

fig=go.Figure()
fig.add_trace(go.Scatter(x=df.date,y=df.close,name='Observed close',line=dict(color='#e2e8f0',width=2)))
for low,high,color,label in [(bands[0],bands[3],'rgba(167,139,250,0.16)','95% scenario band'),
                              (bands[1],bands[2],'rgba(34,211,238,0.25)','80% scenario band')]:
    fig.add_trace(go.Scatter(x=future_dates,y=low,mode='lines',line_width=0,showlegend=False,hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=future_dates,y=high,mode='lines',line_width=0,fill='tonexty',fillcolor=color,name=label))
fig.add_trace(go.Scatter(x=future_dates,y=center,name=f'{SELECTED} forecast',line=dict(color='#22d3ee',width=3)))
fig.add_vline(x=df.date.max().timestamp()*1000,line_dash='dot',line_color='#fbbf24')
fig.update_yaxes(title='USD/share')
finish(fig,'13 • SPCX through December 2026 — conditional scenarios',620)
fig.show(renderer="iframe")
display(forecast.head())
display(forecast.tail())
forecast.to_csv(OUTPUT_DIR/'spcx_forecast_2026.csv',index=False,float_format='%.4f')

One-origin holdout scenario coverage: 80% band 100%; 95% band 100%. Not calibration evidence.


,date,ticker,model,forecast_close,scenario_low_80,scenario_high_80,scenario_low_95,scenario_high_95,forecast_origin,horizon_sessions
0,2026-10-20,SPCX,LastClose,147.9600,141.6578,156.3911,141.1383,161.5570,2026-10-19,1
1,2026-10-21,SPCX,LastClose,147.9600,139.2342,158.3005,136.6156,177.1377,2026-10-19,2
2,2026-10-22,SPCX,LastClose,147.9600,138.1654,159.3616,133.8917,169.6464,2026-10-19,3
3,2026-10-23,SPCX,LastClose,147.9600,137.5259,164.6920,133.0155,179.1245,2026-10-19,4
4,2026-10-26,SPCX,LastClose,147.9600,135.0823,166.9118,131.1852,178.1401,2026-10-19,5


,date,ticker,model,forecast_close,scenario_low_80,scenario_high_80,scenario_low_95,scenario_high_95,forecast_origin,horizon_sessions
46,2026-12-24,SPCX,LastClose,147.9600,109.3457,207.0813,94.9958,250.5911,2026-10-19,47
47,2026-12-28,SPCX,LastClose,147.9600,109.5491,208.4777,95.3264,253.0353,2026-10-19,48
48,2026-12-29,SPCX,LastClose,147.9600,109.1826,208.3621,94.7900,254.6404,2026-10-19,49
49,2026-12-30,SPCX,LastClose,147.9600,108.4301,209.7317,94.7200,254.6162,2026-10-19,50
50,2026-12-31,SPCX,LastClose,147.9600,108.2087,211.0514,94.2270,254.5791,2026-10-19,51


In [20]:
last=forecast.iloc[-1]
outlook=(f"### What the forecast actually says\n\n"
 f"The supplied close moves from **${df.close.iloc[0]:.2f}** to **${df.close.iloc[-1]:.2f}**, a **{total_return:+.2f}%** change. "
 f"The maximum observed closing drawdown is **{eda.drawdown_pct.min():.2f}%**.\n\n"
 f"Validation selected **{SELECTED}**. Its fixed-origin holdout MAE is **${holdout.loc[SELECTED,'MAE_USD']:.2f}**, "
 f"compared with **${holdout.loc['LastClose','MAE_USD']:.2f}** for last-close. "
 f"After refitting to all {len(df)} rows, the December 31 point forecast is **${last.forecast_close:.2f}** "
 f"({(last.forecast_close/df.close.iloc[-1]-1)*100:+.2f}% from the last supplied close). "
 f"The 80% scenario range is **${last.scenario_low_80:.2f}–${last.scenario_high_80:.2f}**; "
 f"the 95% range is **${last.scenario_low_95:.2f}–${last.scenario_high_95:.2f}**.\n\n"
 "These are conditional outputs of this small-sample exercise, not an analyst price target. "
 "The model cannot anticipate earnings, launch outcomes, regulation, capital raising, interest-rate changes or market-wide shocks. "
 "Any of those could move prices outside the illustrated bands. A stable-looking point path can coexist with large uncertainty. "
 "Collect more history, verify corporate-action adjustments, and repeat rolling long-horizon tests before making stronger claims.")
display(Markdown(outlook))
(OUTPUT_DIR/'analysis_summary.md').write_text('# SPCX analysis summary\n\nAuthor: Muhammad Ibrahim Shahrukh\n\n'+outlook,encoding='utf-8')
summary={'author':'Muhammad Ibrahim Shahrukh','rows':len(df),'start':str(df.date.min().date()),
    'end':str(df.date.max().date()),'last_close':float(df.close.iloc[-1]),'close_change_pct':float(total_return),
    'max_drawdown_pct':float(eda.drawdown_pct.min()),'selected_model':SELECTED,
    'holdout_mae':float(holdout.loc[SELECTED,'MAE_USD']),'baseline_holdout_mae':float(holdout.loc['LastClose','MAE_USD']),
    'holdout_rmse':float(holdout.loc[SELECTED,'RMSE_USD']),'forecast_sessions':len(forecast),
    'year_end_point':float(last.forecast_close),'year_end_low80':float(last.scenario_low_80),
    'year_end_high80':float(last.scenario_high_80),'year_end_low95':float(last.scenario_low_95),
    'year_end_high95':float(last.scenario_high_95),'holdout_scenario_coverage80_pct':coverage80,
    'holdout_scenario_coverage95_pct':coverage95}
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

Out[20]: 666


### What the forecast actually says

The supplied close moves from **$160.95** to **$147.96**, a **-8.07%** change. The maximum observed closing drawdown is **-48.78%**.

Validation selected **LastClose**. Its fixed-origin holdout MAE is **$6.91**, compared with **$6.91** for last-close. After refitting to all 89 rows, the December 31 point forecast is **$147.96** (+0.00% from the last supplied close). The 80% scenario range is **$108.21–$211.05**; the 95% range is **$94.23–$254.58**.

These are conditional outputs of this small-sample exercise, not an analyst price target. The model cannot anticipate earnings, launch outcomes, regulation, capital raising, interest-rate changes or market-wide shocks. Any of those could move prices outside the illustrated bands. A stable-looking point path can coexist with large uncertainty. Collect more history, verify corporate-action adjustments, and repeat rolling long-horizon tests before making stronger claims.

<div style="background:#111d35;border-left:6px solid #34d399;padding:18px 24px;border-radius:12px;color:#f1f5f9"><span style="color:#34d399;font-weight:bold">10</span><h2 style="margin:6px 0;color:#fff">Reusable outputs & further work</h2><p style="margin:0">Keep the historical CSV, forecasts, evaluation tables and source notes distinct.</p></div>

**Next investigations:** extend the history as new sessions become available; compare a verified market benchmark; evaluate interval coverage over many independent forecast origins; test robustness to IPO-period exclusion using nested validation; and collect documented corporate actions. Do not add fundamentals without aligning them to their original publication dates.

**Sources and attribution — reviewed October 20, 2026**

1. [Yahoo Finance — SPCX history](https://finance.yahoo.com/quote/SPCX/history/): stated upstream market data source. The uploader provided the export; this notebook does not fetch or silently replace it.
2. [yfinance project](https://github.com/ranaroussi/yfinance) and [download reference](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html): collection tool, independent of Yahoo. Download options can change price adjustment and header structure; the original options are unknown.
3. [Nasdaq — SpaceX listing, June 12, 2026](https://www.nasdaq.com/newsroom/spacex-ipo-rocket-company-launches-historic-ipo): identity and listing context.
4. [Nasdaq Trader — 2026 calendar](https://nasdaqtrader.com/Trader.aspx?id=Calendar): scheduled sessions and early closes.
5. [scikit-learn model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html): metric definitions.

**Reuse:** attribution is required for transparent provenance but does not itself grant market-data redistribution rights. The yfinance software license does not license Yahoo's underlying data. See `DATA_LICENSE_NOTES.md` before choosing a Kaggle data license. No open-data license is asserted here.

**Rendering:** all figures use `fig.show(renderer="iframe")`. Run the notebook after upload to regenerate `iframe_figures`. GitHub's notebook viewer may not display iframe output; the standalone HTML report below includes the Plotly runtime and works in a browser without a CDN.

In [21]:
import html
parts=['<!DOCTYPE html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">',
       '<title>SpaceX SPCX • Market Data Lab</title><style>body{margin:0;background:#071225;color:#e2e8f0;font-family:Arial,sans-serif}main{max-width:1250px;margin:auto;padding:30px}header{padding:30px;background:linear-gradient(120deg,#172950,#44245e);border-radius:18px}section{margin:24px 0;border:1px solid #253556;border-radius:14px;padding:12px}p{line-height:1.6}.note{color:#cbd5e1;white-space:pre-wrap}</style></head><body><main>',
       '<header><p style="color:#67e8f9">MARKET DATA LAB / 2026</p><h1>SpaceX • SPCX</h1><h2>Price history, risk & forecast scenarios</h2><p>Muhammad Ibrahim Shahrukh · Snapshot through October 19, 2026</p></header>',
       '<p class="note">'+html.escape(outlook.replace('### ','').replace('**',''))+'</p>',
       '<p>Source: uploader-provided Yahoo Finance export via yfinance. Adjustment settings unknown. Scenario bands are not calibrated prediction intervals. Review the notebook for methods and caveats.</p>']
for i,figure in enumerate(FIGURES):
    parts.append('<section>'+figure.to_html(full_html=False,include_plotlyjs=(True if i==0 else False),config={'responsive':True})+'</section>')
parts.append('<footer><p>Sources: <a href="https://finance.yahoo.com/quote/SPCX/history/">Yahoo Finance</a> · <a href="https://github.com/ranaroussi/yfinance">yfinance</a> · <a href="https://nasdaqtrader.com/Trader.aspx?id=Calendar">Nasdaq calendar</a></p></footer></main></body></html>')
(OUTPUT_DIR/'spcx_interactive_report.html').write_text(''.join(parts),encoding='utf-8')
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','plotly','scikit-learn','scipy']}
(OUTPUT_DIR/'environment_versions.json').write_text(json.dumps({'python':platform.python_version(),**versions},indent=2))
print(f'Complete: {len(FIGURES)} interactive figures; exports saved to {OUTPUT_DIR.resolve()}')

Complete: 13 interactive figures; exports saved to /workspace/scratch/f237b82f6048/spcx_kaggle_package/reports
